# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [1]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [2]:
#Colocando o caminho dos arquivos aqui
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\RS_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\4302808\4302808.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Criados\APS.gpkg')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\Shapefiles Tomo I\SB_CS_ok.shp')
coluna_nome_bacias = 'Nome'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Criados\popdom_bacia_2022.xlsx'

# Não é necessário mexer no que está abaixo.

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [3]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e transformar csv de domicílios em um arquivo georreferenciado

In [4]:
domparticular = domicilios[domicilios['COD_ESPECIE']==1]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS (escolhi WGS84 = 4326)

In [5]:
bacias = bacias.to_crs(4326)
aps = aps.to_crs(4326)
setores = setores.to_crs(4326) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [6]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [7]:
#contagem de domicílios em cada setor na APS

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

##### Cálculo da população com a densidade e n_pontos criado

In [8]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 26333
A população total na APS em 2022 é de 13372


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [9]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [10]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [11]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [12]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
Nome,,
-,5,9.132424
AR1,409,902.269426
AR2,120,237.822137
AR3,343,705.690534
BT1,228,465.087343
CE1,1289,2607.196927
CE2,378,772.177863
CE3,650,1264.100109
CE4,1439,2827.492396


##### Exportar excel final

In [13]:
bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [14]:
pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

display(bacias_populacao)

A população total na APS em 2022 é de 26333
A população total na APS em 2022 é de 13372


,Domicílios,População
Nome,,
-,5,9.132424
AR1,409,902.269426
AR2,120,237.822137
AR3,343,705.690534
BT1,228,465.087343
CE1,1289,2607.196927
CE2,378,772.177863
CE3,650,1264.100109
CE4,1439,2827.492396
